In [ ]:
%load_ext autoreload
%autoreload 2

### Setup

In [2]:
import os
import sys
sys.path.append(os.path.dirname(os.getcwd())) # Add the parent directory to sys.path

import pandas as pd
import numpy as np

from utils import DATA_DIR
from dataloader import get_nd_array, get_slice

from download.weka import pull_predictions_from_weka

In [ ]:
# pull_predictions_from_weka()

File already exists and matches size: /Users/dhei/ai2/new-evals/analysis/data/all_aws_predictions.parquet


In [4]:
df = pd.read_parquet(f'{DATA_DIR}/all_aws_predictions.parquet')

In [5]:
models = df.index.get_level_values('model').unique().to_list()
tasks  = df.index.get_level_values('task').unique().to_list()
mixes  = df.index.get_level_values('mix').unique().to_list()

tasks = [t for t in tasks if 'mmlu' not in t]

# print(models)
# print(tasks)
# print(mixes)

In [6]:
# models, scores = get_nd_array(df, 'model', 'acc_per_char', task='arc_challenge:distractors')
# models, scores = get_nd_array(df, ['model', 'mix'], 'acc_per_char', task='arc_challenge:distractors')

### Measure 1: Predictability (Rel Error @ 7B)

In [7]:
# ladder_models = [m for m in models if 'peteish-moreeval' in m]

### Measure 2: Seperability (% sig @ 1B)

In [ ]:
from stats import compute_significance

In [ ]:
TASKS = ['arc_easy'] # ['arc_easy', 'boolq', 'hellaswag']

sig_results, _ = compute_significance(df, 'peteish-moreeval-1B-5xC', 'acc_per_char', tasks=TASKS, do_plot=False)

# print('% of significant comparisons at each model scale (improvement over :rc in parentheses):')
# display_improvement_diff(sig_results, index=model_order)

### Measure 3: Smoothness (Total Variation @ 1B)

In [ ]:
from stats import calc_total_variation

In [ ]:
# Problem: HellaSwag has duplicate question ids!!
slices = get_slice(df, model='peteish-moreeval-1B-5xC', task='hellaswag')

# display(slices)

# pivoted = slices.pivot(index='native_id', columns=['step'], values='acc_per_char')

# pivoted
# slices

slices['native_id'].value_counts().value_counts()

In [ ]:
tv_results = pd.DataFrame(index=tasks, columns=['total_variation'])

for task in tasks:
    if 'arc' not in task: continue
    
    models, scores = get_nd_array(df, 'step', 'acc_per_char', model='peteish-moreeval-1B-5xC', task=task)

    if len(scores) <= 1: continue

    acc = scores.mean(axis=1)
    
    tv_results.loc[task, 'total_variation'] = calc_total_variation(acc, improvement=True) * 100

display(tv_results)